In [ ]:
import pandas as pd
import numpy as np

#Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold

#Model Machine Learning
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error,mean_squared_error,root_mean_squared_error,mean_absolute_percentage_error,r2_score
import warnings
warnings.filterwarnings('ignore',category=UserWarning)


## 1. Simple Linear Regression

In [28]:
df = pd.read_csv('../0.dataset/dataset_California_HousePrice_clean.csv')
df_x = df.drop(columns='median_house_value')
df_y = df['median_house_value']

In [29]:
feature_numerik = df.select_dtypes(include=np.number).drop(columns='median_house_value').columns
feauture_categori = df.select_dtypes(include='object').columns
preprocessor = ColumnTransformer(
    transformers=[
        ('numerik_scaler',RobustScaler(),feature_numerik),
        ('categori_encoding', OneHotEncoder(), feauture_categori) # Jika ketemu kategori baru saat testing, kolom dummy-nya otomatis diisi 0 semua (diabaikan)
    ]
)

In [30]:
model_simpleRegression = Pipeline([
    ('prerocessing',preprocessor),
    ('model_regression',LinearRegression())
])

In [31]:
def evaluasi_cross_validation(cv_results):
    metrics = {
        'MAE': ('train_mae', 'test_mae', True),       # (Kunci Train, Kunci Test, Apakah Negatif Error?)
        'MSE': ('train_mse', 'test_mse', True),
        'RMSE': ('train_rmse', 'test_rmse', True),
        'MAPE': ('train_mape', 'test_mape', True),
        'R2 Score': ('train_r2', 'test_r2', False)
    }
    
    hasil = {}
    for nama_metrik, (kunci_train, kunci_test, is_negative_error) in metrics.items():
        # Ambil rata-rata nilai dari seluruh fold
        nilai_train = cv_results[kunci_train].mean()
        nilai_test = cv_results[kunci_test].mean()
        
        # Jika metrik bawaan sklearn bernilai negatif (neg_*), kalikan dengan -1
        if is_negative_error:
            nilai_train = -nilai_train
            nilai_test = -nilai_test
            
        hasil[nama_metrik] = [nilai_train, nilai_test]
        
    df_hasil = pd.DataFrame(hasil, index=['Train', 'Test']).T
    return df_hasil

In [32]:
scoring_metrics = {
    "r2": "r2",
    "mae": "neg_mean_absolute_error", 
    "mse": "neg_mean_squared_error",
    "rmse": "neg_root_mean_squared_error",
    "mape": "neg_mean_absolute_percentage_error"
}
# Acak urutan baris data secara internal sebelum dibagi menjadi 5 bagian suoaya tidak nan
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_validate(model_simpleRegression,df_x,df_y,cv=kf,scoring=scoring_metrics,return_train_score=True,n_jobs=-1,)

df_evaluasi = evaluasi_cross_validation(cv_results)
print("=== HASIL EVALUASI 5-FOLD CROSS VALIDATION ===")
print(df_evaluasi.round(3))

=== HASIL EVALUASI 5-FOLD CROSS VALIDATION ===
                 Train          Test
MAE       3.593592e+04  3.627576e+04
MSE       2.660901e+09  2.719949e+09
RMSE      5.156719e+04  5.191261e+04
MAPE      2.200000e-01  2.230000e-01
R2 Score  7.140000e-01  7.080000e-01
